# Solar Utility Agent — Training notebook
Downloads ~3 months of hourly Open-Meteo Archive weather + matching NASA POWER
daily irradiance, labels days by irradiance quantiles, trains a 3-feature
logistic regression, prints the learned constants, and saves `solar_model.pkl`.

Honesty note: the Archive API does not expose `precipitation_probability`, so
training uses archive `precipitation` (mm). Inference uses forecast
`precipitation_probability` (%). The logistic regression absorbs the scale
difference, but this mismatch is documented in the report.

In [ ]:
# !pip -q install joblib scikit-learn pandas numpy requests

import joblib
import numpy as np
import pandas as pd
import requests
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             classification_report)

## 1. Config

In [ ]:
LAT, LON = 4.3596, 100.9761
START, END = "2024-01-01", "2024-03-31"   # ~3 months
FEATURES = ["hour", "ghi", "rain"]
CLASSES = ["GOOD", "MARGINAL", "NO"]

## 2. Pull Open-Meteo Archive (hourly GHI + precipitation)

In [ ]:
r = requests.get(
    "https://archive-api.open-meteo.com/v1/archive",
    params={
        "latitude": LAT, "longitude": LON,
        "start_date": START, "end_date": END,
        "hourly": "shortwave_radiation,precipitation",
        "timezone": "auto",
    }, timeout=60,
)
r.raise_for_status()
om = r.json()
df = pd.DataFrame({
    "time": pd.to_datetime(om["hourly"]["time"]),
    "ghi":  om["hourly"]["shortwave_radiation"],
    "rain": om["hourly"]["precipitation"],
})
df["date"] = df["time"].dt.date
df["hour"] = df["time"].dt.hour
print(df.head(), df.shape)

## 3. Pull NASA POWER daily irradiance (kWh/m²/day)

In [ ]:
r = requests.get(
    "https://power.larc.nasa.gov/api/temporal/daily/point",
    params={
        "parameters": "ALLSKY_SFC_SW_DWN",
        "community": "RE",
        "longitude": LON, "latitude": LAT,
        "start": START.replace("-", ""), "end": END.replace("-", ""),
        "format": "JSON",
      
    }, timeout=60,
)
r.raise_for_status()
pw = r.json()["properties"]["parameter"]["ALLSKY_SFC_SW_DWN"]
pw_df = pd.DataFrame({
    "date": pd.to_datetime(list(pw.keys())).date,
    "daily_irr": list(pw.values()),
})
pw_df = pw_df[pw_df["daily_irr"] > -900]          # drop NASA -999 fill
print(pw_df.head(), pw_df.shape)

## 4. Derive quantiles from the *daily* distribution, then label days
`NO < q33 ≤ MARGINAL < q66 ≤ GOOD`

In [ ]:
# 1. Scale precipitation (mm) to roughly match the 0-100% forecast scale
df["rain"] = df["rain"] * 10.0

# 2. Merge hourly Open-Meteo with daily NASA POWER
merged = df.merge(pw_df, on="date", how="inner")

# 3. Derive quantiles from the daily irradiance distribution
q33 = merged["daily_irr"].quantile(0.33)
q66 = merged["daily_irr"].quantile(0.66)
print(f"q33 = {q33:.3f} kWh/m²/day, q66 = {q66:.3f} kWh/m²/day")

# 4. Label helpers
def label_day(daily_irr):
    if daily_irr >= q66:  return "GOOD"
    if daily_irr >= q33:  return "MARGINAL"
    return "NO"

def assign_label(row):
    if row["ghi"] == 0:
        return "NO"
    return label_day(row["daily_irr"])

# 5. Apply to every hour
merged["label"] = merged.apply(assign_label, axis=1)
print(merged["label"].value_counts())

## 5. Train / test split (Jan–Feb train, Mar test)

In [ ]:
merged["month"] = merged["time"].dt.month
train = merged[merged["month"].isin([1, 2])]
test  = merged[merged["month"] == 3]
X_tr, y_tr = train[FEATURES].fillna(0), train["label"]
X_te, y_te = test[FEATURES].fillna(0),  test["label"]
print("train:", X_tr.shape, "test:", X_te.shape)

## 6. Fit logistic regression

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_tr, y_tr)
print("classes:", list(model.classes_))

## 7. Honest evaluation

In [ ]:
y_pred = model.predict(X_te)
print("accuracy :", round(accuracy_score(y_te, y_pred), 4))
print("macro-F1 :", round(f1_score(y_te, y_pred, average="macro"), 4))
print(confusion_matrix(y_te, y_pred, labels=CLASSES))
print(classification_report(y_te, y_pred, labels=CLASSES))

coef = pd.DataFrame(model.coef_, index=model.classes_, columns=FEATURES)
coef["intercept"] = model.intercept_
print(coef.round(3))

## 8. Learned FLOOR and GOOD (replaces hand-picked 0.30 / 0.60)
FLOOR = mean utility of hours labelled MARGINAL ·
GOOD  = mean utility of hours labelled GOOD

In [ ]:
def utility_from_proba(p):
    d = dict(zip(model.classes_, p))
    return 1.0*d.get("GOOD", 0) + 0.5*d.get("MARGINAL", 0) + 0.0*d.get("NO", 0)

u_tr = np.array([utility_from_proba(p) for p in model.predict_proba(X_tr)])
FLOOR = float(u_tr[y_tr == "MARGINAL"].mean())
GOOD  = float(u_tr[y_tr == "GOOD"].mean())
print(f"FLOOR = {FLOOR:.3f}, GOOD = {GOOD:.3f}")

## 9. Learned-vs-hand-picked comparison

In [ ]:
p_te = model.predict_proba(X_te)
u_te = np.array([utility_from_proba(p) for p in p_te])
new_verdict = np.array(model.classes_)[np.argmax(p_te, axis=1)]
old_verdict = np.where(u_te >= 0.60, "GOOD",
              np.where(u_te >= 0.30, "MARGINAL", "NO"))
agree = (old_verdict == new_verdict).mean()
print(f"Agreement (learned vs hand-picked) on test hours: {agree:.1%}")

## 10. Save artifact

In [ ]:
artifact = {
    "model":      model,
    "features":   FEATURES,
    "classes":    list(model.classes_),
    "floor":      FLOOR,
    "good":       GOOD,
    "q33":        float(q33),
    "q66":        float(q66),
    "trained_on": f"{START}..{END}",
    "location":   {"lat": LAT, "lon": LON},
}
joblib.dump(artifact, "solar_model.pkl")
print("saved solar_model.pkl")